# Cross-Africa Satellite Prediction: Leave-One-Country-Out Evaluation (NBK 3)

This notebook executes the core cross-Africa test. For every African country in the dataset and every outcome, the model is trained on all other African countries and evaluated on the held-out country's ADM1 regions. This mimics the real policy scenario of predicting development indicators for a country without a recent survey, and provides the per-country R²/MAE/Spearman rank correlation table that downstream analyses delve into.

---

### Import statements

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from scipy import stats
import helpers

---

### Loading the merged dataset

Features are the population-weighted MOSAIKS embeddings (4,000 dimensions per region), and labels are the augmented table from `labels_augmented.csv`. The two are merged on `gdlcode` to produce one row per region with both predictors and outcomes.

In [ ]:
labels = pd.read_csv(os.path.join(helpers.PROC, "labels_augmented.csv"))
feat = pd.read_parquet(os.path.join(helpers.PROC, "features_population_weighted.parquet"))
feat = feat[["gdlcode"] + [c for c in feat.columns if c.startswith("X_")]]
feat_cols = [c for c in feat.columns if c.startswith("X_")]

df = labels.merge(feat, on="gdlcode", how="inner")
print(f"Merged dataset: {len(df)} regions, {df['isocode3'].nunique()} countries, {len(feat_cols)} features")

Merged dataset: 538 regions, 53 countries, 4000 features


538 African ADM1 regions across 53 countries carry both MOSAIKS features and at least one development outcome.

---

### LOCO ridge with per-country progress

For each (country, outcome) pair the ridge penalty is tuned by 5-fold GroupKFold cross-validation grouped by country (so no country appears in both training and validation folds). Two baselines are computed alongside: predicting the training-set mean, and ridge regression using only urban share as a single feature.

In [6]:
all_metrics, all_predictions = [], []
t_total = time.time()

for outcome in helpers.OUTCOMES:
    if outcome not in df.columns:
        continue
    sub = df.dropna(subset=[outcome]).copy()
    countries = sorted(sub["isocode3"].unique())
    print(f"\nOutcome: {outcome}  ({len(sub)} regions, {len(countries)} countries)", flush=True)
    t_out = time.time()

    for j, ctry in enumerate(countries, 1):
        train = sub[sub["isocode3"] != ctry]
        test  = sub[sub["isocode3"] == ctry]
        if len(train) < 20 or len(test) < 2 or train["isocode3"].nunique() < 2:
            continue

        X_tr, y_tr = train[feat_cols].values.astype(float), train[outcome].values.astype(float)
        X_te, y_te = test[feat_cols].values.astype(float),  test[outcome].values.astype(float)
        sc = StandardScaler()
        X_tr_sc, X_te_sc = sc.fit_transform(X_tr), sc.transform(X_te)

        y_pred, alpha = helpers.fit_ridge_cv(X_tr_sc, y_tr, train["isocode3"].values, X_te_sc)
        r2, mae, rho  = helpers.compute_metrics(y_te, y_pred)

        tm = y_tr.mean()
        r2_mean  = r2_score(y_te, np.full_like(y_te, tm)) if len(np.unique(y_te)) > 1 else np.nan
        mae_mean = mean_absolute_error(y_te, np.full_like(y_te, tm))
        if train["urban"].notna().sum() > 10 and test["urban"].notna().sum() > 0:
            u_tr = train[["urban"]].fillna(train["urban"].median()).values
            u_te = test[["urban"]].fillna(train["urban"].median()).values
            scu = StandardScaler()
            m_u  = Ridge(alpha=1.0); m_u.fit(scu.fit_transform(u_tr), y_tr)
            y_u  = m_u.predict(scu.transform(u_te))
            r2_u, mae_u = helpers.compute_metrics(y_te, y_u)[0], mean_absolute_error(y_te, y_u)
        else:
            r2_u = mae_u = np.nan

        all_metrics.append({
            "outcome": outcome, "isocode3": ctry, "country": test["country"].iloc[0],
            "geo_region": test["geo_region"].iloc[0], "n_regions": len(test), "n_train": len(train),
            "alpha": alpha, "r2": r2, "mae": mae, "spearman_rho": rho,
            "r2_mean_baseline": r2_mean, "mae_mean_baseline": mae_mean,
            "r2_urban_baseline": r2_u, "mae_urban_baseline": mae_u,
            "y_true_mean": y_te.mean(), "y_true_std": y_te.std(),
            "y_train_mean": y_tr.mean(), "y_train_std": y_tr.std(),
        })
        for i, row in enumerate(test.itertuples()):
            all_predictions.append({
                "outcome": outcome, "isocode3": ctry, "country": test["country"].iloc[0],
                "gdlcode": row.gdlcode, "region": row.region,
                "y_true": y_te[i], "y_pred": y_pred[i],
                "region_pop": getattr(row, "region_pop", np.nan),
            })
        print(f"  [{j:>2}/{len(countries)}] {ctry}  R²={r2:+.3f}  MAE={mae:.3f}  ρ={rho:+.3f}", flush=True)

    print(f"  Outcome done in {time.time()-t_out:.1f}s", flush=True)

print(f"\nTotal: {time.time()-t_total:.1f}s")


Outcome: iwi  (459 regions, 43 countries)


  [ 1/43] AGO  R²=-0.295  MAE=10.170  ρ=+0.168
  [ 2/43] BDI  R²=-2.053  MAE=5.895  ρ=+0.600
  [ 3/43] BEN  R²=-2.972  MAE=9.970  ρ=+0.600
  [ 4/43] BFA  R²=+0.269  MAE=5.109  ρ=+0.674
  [ 5/43] CAF  R²=-0.164  MAE=10.627  ρ=+0.486
  [ 6/43] CIV  R²=-1.761  MAE=8.420  ρ=+0.539
  [ 7/43] CMR  R²=+0.160  MAE=9.019  ρ=+0.491
  [ 8/43] COD  R²=-0.368  MAE=12.655  ρ=+0.336
  [ 9/43] COG  R²=-0.414  MAE=14.166  ρ=+0.273
  [10/43] COM  R²=-64.906  MAE=18.331  ρ=-0.500
  [11/43] DZA  R²=-79.481  MAE=6.042  ρ=-0.107
  [12/43] EGY  R²=-118.440  MAE=25.101  ρ=+0.328
  [13/43] ETH  R²=+0.262  MAE=11.369  ρ=+0.491
  [14/43] GAB  R²=-4.706  MAE=15.914  ρ=+0.433
  [15/43] GHA  R²=+0.495  MAE=3.677  ρ=+0.782
  [16/43] GIN  R²=+0.636  MAE=7.246  ρ=+0.714
  [17/43] GMB  R²=+0.212  MAE=9.986  ρ=+0.548
  [18/43] GNB  R²=-2.366  MAE=9.638  ρ=+0.250
  [19/43] KEN  R²=-1.056  MAE=9.142  ρ=+0.690
  [20/43] LBR  R²=-6.513  MAE=16.487  ρ=+0.711
  [21/43] LBY  R²=-11.501  MAE=10.118  ρ=+1.000
  [22/43] LSO  R²=-

The loop produces one row per (country × outcome) with metrics and baselines, plus a per-region prediction row for downstream analysis.

---

### Saving outputs

Per-country metrics and per-region predictions are saved separately:
- `loco_metrics.csv` has one row per (country × outcome) with R², MAE, Spearman ρ, and both baseline metrics. This is the main input for the explanation regression and the distribution shift test.
- `loco_predictions.csv` stores every held-out prediction with its true value and region population, ready for policy-counterfactual analysis in the planned Nigeria case study.

In [8]:
metrics_df = pd.DataFrame(all_metrics)
preds_df = pd.DataFrame(all_predictions)
metrics_df.to_csv(os.path.join(helpers.OUT, "loco_metrics.csv"), index=False)
preds_df.to_csv(os.path.join(helpers.OUT, "loco_predictions.csv"), index=False)

print("Median R² across countries by outcome:")
metrics_df.groupby("outcome")["r2"].agg(["median","mean","min","max"]).round(3)

Median R² across countries by outcome:


,median,mean,min,max
outcome,,,,
dtp3age1,-2.314,-242.881,-4712.651,0.110
gap_eye,-1.480,-2.704,-15.899,0.507
iwi,-0.414,-8.890,-118.440,0.807
iwipov70,-1.052,-12.823,-150.578,0.885
shdi,-0.944,-11.673,-249.967,0.813
stunting,-0.870,-3.567,-42.931,0.451
u5mort,-0.696,-3.721,-54.799,0.275


A cross-outcome median R² summary is printed as a sanity check.

---